In [ ]:
from pathlib import Path

import pandas as pd
import plotly.io as pio

from alpaca.plotting import (
    load_chr_table,
    load_mutation_table,
    prepare_driver_mutations,
    plot_cpn_per_clone,
    plot_heatmap_with_tree,
    plot_sample_level_copy_numbers,
)
from alpaca.utils import read_tree_json

pio.renderers.default = "notebook_connected"


In [ ]:
INPUT_DIR = Path(r"/Users/piotrpawlik/Documents/GitHub/ALPACA-model/examples/example_cohort/input/LTX0000-Tumour1")
OUTPUT_DIR = Path(r"/Users/piotrpawlik/Documents/GitHub/ALPACA-model/tests/correct_input/output/LTX0000-Tumour1")
TREE_PATH = Path(r"/Users/piotrpawlik/Documents/GitHub/ALPACA-model/examples/example_cohort/input/LTX0000-Tumour1/tree_paths.json")
CP_TABLE_PATH = Path(r"/Users/piotrpawlik/Documents/GitHub/ALPACA-model/examples/example_cohort/input/LTX0000-Tumour1/cp_table.csv")
ALPACA_OUTPUT_PATH = Path(r"/Users/piotrpawlik/Documents/GitHub/ALPACA-model/tests/correct_input/output/LTX0000-Tumour1/ALPACA_output_LTX0000-Tumour1.csv")
SAMPLE_TABLE_PATH = INPUT_DIR / "ALPACA_input_table.csv"

HEATMAP_PALETTE = "classic"
GENOME_BUILD = "hg19"

chr_table = load_chr_table(genome_build=GENOME_BUILD)
tree = read_tree_json(str(TREE_PATH))
alpaca_output = pd.read_csv(ALPACA_OUTPUT_PATH)
cp_table = pd.read_csv(CP_TABLE_PATH).set_index("clone")

sample_table = None
if SAMPLE_TABLE_PATH.exists():
    sample_table = pd.read_csv(SAMPLE_TABLE_PATH)
else:
    print("Sample-level input table not found; sample-level plots will be skipped.")

mutation_table = load_mutation_table(INPUT_DIR)
tumour_id = alpaca_output.tumour_id.iloc[0]
driver_mutations = prepare_driver_mutations(mutation_table, tumour_id, chr_table)


if driver_mutations is None:
    print("Driver mutation annotations unavailable; plots will omit mutation markers.")
else:
    print(f"Driver mutation annotations loaded")


In [ ]:
heatmap_A = plot_heatmap_with_tree(
    tree=tree,
    alpaca_output=alpaca_output.copy(),
    cp_table=cp_table,
    chr_table=chr_table,
    driver_mutations=driver_mutations,
    allele="A",
    heatmap_palette=HEATMAP_PALETTE,
)
heatmap_A.show()


In [ ]:
heatmap_B = plot_heatmap_with_tree(
    tree=tree,
    alpaca_output=alpaca_output.copy(),
    cp_table=cp_table,
    chr_table=chr_table,
    driver_mutations=driver_mutations,
    allele="B",
    heatmap_palette=HEATMAP_PALETTE,
)
heatmap_B.show()


In [ ]:
cn_changes = plot_cpn_per_clone(
    tree=tree,
    alpaca_output=alpaca_output.copy(),
    cp_table=cp_table,
    chr_table=chr_table,
    driver_mutations=driver_mutations,
    max_cpn_cap=8,
)
cn_changes.show()


In [ ]:
if sample_table is None or sample_table.empty:
    print("Sample-level input table missing; skipping sample-level copy-number plots.")
else:
    sample_cpn_A = plot_sample_level_copy_numbers(
        sample_table=sample_table,
        chr_table=chr_table,
        allele="A",
        max_cpn_cap=8,
    )
    sample_cpn_A.show()

    sample_cpn_B = plot_sample_level_copy_numbers(
        sample_table=sample_table,
        chr_table=chr_table,
        allele="B",
        max_cpn_cap=8,
    )
    sample_cpn_B.show()
